## 06-deppagent/test.ipynb

1. csv 파일을 받아서
1. 데이터 분석을 진행하고
3. 슬랙으로 결과를 보냄

```sh
uv pip install deepagents slack-sdk langchain-daytona langchain-openai
```

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from daytona import Daytona
from langchain_daytona import DaytonaSandbox

sandbox = Daytona().create()
backend = DaytonaSandbox(sandbox=sandbox)

In [3]:
result = backend.execute('ls -a')

In [4]:
print(result.output)

.
..
.bash_logout
.bashrc
.daytona
.face
.face.icon
.profile
.zshrc


In [5]:
import csv
import io 

data = [
    ["Date", "Product", "Units Sold", "Revenue"],
    ["2025-08-01", "Widget A", 10, 250],
    ["2025-08-02", "Widget B", 5, 125],
    ["2025-08-03", "Widget A", 7, 175],
    ["2025-08-04", "Widget C", 3, 90],
    ["2025-08-05", "Widget B", 8, 200],
]

buf = io.StringIO()
writer = csv.writer(buf)
writer.writerows(data)
csv_bytes = buf.getvalue().encode('utf-8')
buf. close()

# 파일 저장(지금 우리는 필요 없음)
with open('./sales.csv', 'wb') as f:
    f.write(csv_bytes)

# sandbox 에 업로드
backend.upload_files(
    [
        ('/home/daytona/data/sales_data.csv', csv_bytes)
        ]
)

[FileUploadResponse(path='/home/daytona/data/sales_data.csv', error=None)]

## Slack
- Agent 가 사용할 Slack Messaging Tool 생성

In [6]:
import os
from slack_sdk import WebClient

SLACK_BOT_TOKEN = os.getenv('SLACK_BOT_TOKEN')

client = WebClient(token=SLACK_BOT_TOKEN)

# 봇이 접근 가능한 모든 채널 리스트
res = client.conversations_list()
for ch in res['channels']:
    print(ch['name'], ch['id'])

social_channel_id = 'D0A4101M31C'

client.chat_postMessage(
    channel=social_channel_id,
    text='hello'
)

소셜 C0A40SJTY8N
slack-전체 C0A46FL3TGU
새-채널 C0A4FRYA3QR


In [ ]:
import os
from slack_sdk import WebClient
from langchain.tools import tool

SLACK_TOKEN = os.getenv('SLACK_BOT_TOKEN')
slack_client = WebClient(token=SLACK_TOKEN)

@tool(parse_docstring=True) # LLM에게 Tool 설명을 docstring 에서 제공
def send_slack_message(text: str, file_path: str | None = None) -> str:
    """메세지 전송, 경우에 따라 이미지같은 파일을 첨부함
    
    Args:
        text: (str) 메세지의 내용
        file_path: (str) 파일시스템 상 첨부파일의 경로
    """
    # 첨부 파일 없으면
    social_channel_id = 'D0A4101M31C'
    if not file_path:
        slack_client.chat_postMessage(channel=social_channel_id, text=text)
    else:
        fp = backend.download_files([file_path])
        slack_client.files_upload_v2(
            channel=social_channel_id,
            content=fp[0].content,
            initial_comment=text,
        )
    return 'Message sent.'

## Build Deep Agent

In [8]:
import uuid

from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model='openai:gpt-5-mini',
    tools=[send_slack_message],
    backend=backend,
    checkpointer=checkpointer,
)

thread_id = str(uuid.uuid4())
config = {'configurable': {'thread_id': thread_id}}

In [ ]:
input_message = {
    'role': 'user',
    'content': '''
    현재 폴더 안에 ./data/sales_data.csv 파일을 분석하고, 시각화 해줘.
    다 끝나면 분석결과와 시각화 이미지를 Slack 메세지로 보내줘.
    '''
}

for step in agent.stream(
    {"messages": [input_message]},
    config,
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_0fdeb9b7dce7207b0069afb6878f5081948e582df04af76115', 'summary': [], 'type': 'reasoning'}, {'arguments': '{"todos":[{"content":"Inspect ./data/sales_data.csv (list directory, read first lines)","status":"in_progress"},{"content":"Run analysis script to generate summary and plots (create script, execute)","status":"pending"},{"content":"Send analysis results and plots to Slack (attach zip of images)","status":"pending"}]}', 'call_id': 'call_ijGDSR1xofgDoCpENWFl26MW', 'name': 'write_todos', 'type': 'function_call', 'id': 'fc_0fdeb9b7dce7207b0069afb69dc5e88194ba9abccfcb646553', 'status': 'completed'}]
Tool Calls:
  write_todos (call_ijGDSR1xofgDoCpENWFl26MW)
 Call ID: call_ijGDSR1xofgDoCpENWFl26MW
  Args:
    todos: [{'content': 'Inspect ./data/sales_data.csv (list directory, read first lines)', 'status': 'in_progress'}, {'content': 'Run analysis script to generate summary and plots (create script

In [ ]:

input_message = {
    'role': 'user',
    'content': '''
    채널은 기본 세팅 되어있음. 2번 데이터 시각화 해서 슬랙 메시지로 보내줘
    '''
}

for step in agent.stream(
    {"messages": [input_message]},
    config,
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()